In [1]:
import json

import requests
from bs4 import BeautifulSoup


In [2]:
url = 'https://remotive.com/api/remote-jobs'
response_remotive = requests.get(url)

if response_remotive.status_code == 200:
    data_remotive = response_remotive.json()
    jobs_remotive = data_remotive['jobs']
else:
    print(f'Failed to fetch jobs. Status code: {response_remotive.status_code}')

In [3]:
def clean_job_description(raw_description):
    html_tree = BeautifulSoup(raw_description, 'lxml')
    description = html_tree.get_text()
    description = description.splitlines()
    description = [line.strip() for line in description if line.strip()]
    return '\n'.join(description)

In [4]:
def map_job_remotive(job):
    return {
        'source': 'remotive',
        'source_job_id': str(job['id']),
        'url': job['url'],
        'title': job['title'],
        'company_name': job['company_name'],
        'category': job['category'],
        'tags': json.dumps(job['tags']),
        'job_type': job['job_type'],
        'salary': job['salary'] or None,
        'description': clean_job_description(job['description']),
        'company_logo_url': job['company_logo_url'],
        'location': job['candidate_required_location'],
        'published_at': job['publication_date'],
    }

In [5]:
import os

from dotenv import load_dotenv

load_dotenv()

X_API_Key = os.getenv("X-API-Key")

headers = {
    "X-API-Key": X_API_Key,
}

response_ninja = requests.get('https://api.openwebninja.com/jsearch/search-v2', headers=headers, params={"query": "python developer"})

In [6]:
if response_ninja.status_code == 200:
    data_ninja = response_ninja.json()
    jobs_ninja = data_ninja['data']['jobs']
else: 
    print(f"Error: {response_ninja.status_code} - {response_ninja.text}")

In [7]:
def map_job_ninja(job):
    return {
        'source': 'openwebninja',
        'source_job_id': str(job['job_id']),
        'url': job['job_apply_link'],
        'title': job['job_title'],
        'company_name': job['employer_name'],
        'category': None,
        'tags': None,
        'job_type': job['job_employment_type'],
        'salary': job['job_salary'] or None,
        'description': clean_job_description(job['job_description']),
        'company_logo_url': job['employer_logo'],
        'location': job['job_location'],
        'published_at': job['job_posted_at_datetime_utc'],
    }

In [9]:
import sqlite3
from datetime import datetime, timezone

conn = sqlite3.connect('jobs.db')
fetched_at = datetime.now(timezone.utc).isoformat() 

# raw table 
try:
    with conn:
        for job in jobs_remotive:
            source = 'remotive'
            source_job_id = str(job['id'])
            payload = json.dumps(job)
            conn.execute(
                "INSERT OR IGNORE INTO raw_jobs (source, source_job_id, payload, fetched_at) VALUES (?, ?, ?, ?)",
                (source, source_job_id, payload, fetched_at)
            )
        for job in jobs_ninja:
            source = 'openwebninja'
            source_job_id = str(job['job_id'])
            payload = json.dumps(job)
            conn.execute(
                "INSERT OR IGNORE INTO raw_jobs (source, source_job_id, payload, fetched_at) VALUES (?, ?, ?, ?)",
                (source, source_job_id, payload, fetched_at)
            )
        for job in jobs_remotive:
            mapped_job = map_job_remotive(job)
            conn.execute(
                "INSERT OR IGNORE INTO jobs (source, source_job_id, url, title, company_name, category, tags, job_type, salary, description, company_logo_url, location, published_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (
                    mapped_job['source'],
                    mapped_job['source_job_id'],
                    mapped_job['url'],
                    mapped_job['title'],
                    mapped_job['company_name'],
                    mapped_job['category'],
                    mapped_job['tags'],
                    mapped_job['job_type'],
                    mapped_job['salary'],
                    mapped_job['description'],
                    mapped_job['company_logo_url'],
                    mapped_job['location'],
                    fetched_at,
                )
            )
        for job in jobs_ninja:
            mapped_job = map_job_ninja(job)
            conn.execute(
                "INSERT OR IGNORE INTO jobs (source, source_job_id, url, title, company_name, category, tags, job_type, salary, description, company_logo_url, location, published_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (
                    mapped_job['source'],
                    mapped_job['source_job_id'],
                    mapped_job['url'],
                    mapped_job['title'],
                    mapped_job['company_name'],
                    mapped_job['category'],
                    mapped_job['tags'],
                    mapped_job['job_type'],
                    mapped_job['salary'],
                    mapped_job['description'],
                    mapped_job['company_logo_url'],
                    mapped_job['location'],
                    fetched_at,
                )
            )
finally:
    conn.close()